# 03_mobilenetv3_small.ipynb

Fine-tune MobileNetV3-Small trên split đã khóa, chọn checkpoint theo macro f1 trên val. Tập test không được mở. Notebook này chỉ chạy trên Kaggle.

Cell kế tiếp kéo code từ GitHub. Nhánh nằm ở biến `GIT_BRANCH`.

Hướng dẫn tài khoản, dataset, Internet, GPU và Save Version nằm ở README, mục Chạy trên Kaggle.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

GIT_BRANCH = "baseline-MobileNetV3-Small"
REPO_URL = "https://github.com/meo225/durian-leaf-disease-robustness.git"
REPO_DIR = Path("/kaggle/working") / "durian-leaf-disease-robustness"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(
        ["git", *args],
        cwd=cwd,
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )


if not (REPO_DIR / ".git").is_dir():
    git(["clone", "--branch", GIT_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=REPO_DIR)
    git(["checkout", GIT_BRANCH], cwd=REPO_DIR)
    git(["reset", "--hard", f"origin/{GIT_BRANCH}"], cwd=REPO_DIR)

matches = []
seen = set()
for path in Path("/kaggle/input").rglob("DLD_FinalDataset_224_spit"):
    resolved = str(path.resolve())
    if path.is_dir() and resolved not in seen:
        seen.add(resolved)
        matches.append(path)

if not matches:
    raise FileNotFoundError(
        "Chưa gắn dataset cthng123/durian-leaf-disease-dataset. Xem README, mục Chạy trên Kaggle."
    )

image_root = matches[0].parent
link = REPO_DIR / "datasets" / "durian-ldd"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.is_file():
    link.unlink()
elif link.exists():
    link.rmdir()
link.symlink_to(image_root, target_is_directory=True)

if not (link / "DLD_FinalDataset_224_spit").is_dir():
    raise FileNotFoundError(f"Không thấy ảnh tại {link}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True
).strip()
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
).strip()
print(f"Nhánh: {branch}")
print(f"Commit: {commit}")
print(f"Ảnh: {link / 'DLD_FinalDataset_224_spit'}")

## Cấu hình

In [ ]:
import json
from datetime import datetime, timezone

import pandas as pd
import yaml

from src.utils.runtime import describe_device, get_device, git_commit, set_seed

pipeline_cfg = yaml.safe_load((REPO_DIR / "configs" / "pipeline.yaml").read_text(encoding="utf-8"))
train_cfg = yaml.safe_load((REPO_DIR / "configs" / "mobilenetv3_small.yaml").read_text(encoding="utf-8"))

SEED = pipeline_cfg["seed"]
CLASSES = pipeline_cfg["classes"]
RUN_NAME = f"{train_cfg['model']}_seed{SEED}"

OUTPUTS = REPO_DIR / "outputs"
CHECKPOINT_PATH = OUTPUTS / "checkpoints" / f"{RUN_NAME}.pt"
HISTORY_PATH = OUTPUTS / "logs" / f"{RUN_NAME}_history.csv"
RESULT_PATH = OUTPUTS / "results" / f"{RUN_NAME}.json"
CURVES_PATH = OUTPUTS / "figures" / f"{RUN_NAME}_curves.png"
CONFUSION_PATH = OUTPUTS / "figures" / f"{RUN_NAME}_val_confusion.png"
for path in (CHECKPOINT_PATH, HISTORY_PATH, RESULT_PATH, CURVES_PATH):
    path.parent.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
device = get_device()
print(f"Lần chạy: {RUN_NAME}")
print(f"Thiết bị: {describe_device(device)}")

## Dữ liệu

In [ ]:
from src.data.dataloaders import build_dataloaders

loaders = build_dataloaders(pipeline_cfg, include_test=False)

assert set(loaders) == {"train", "val"}
assert len(loaders["train"].dataset) == pipeline_cfg["expected_train_count"]
assert len(loaders["val"].dataset) == pipeline_cfg["expected_val_count"]

pd.DataFrame({name: loader.dataset.split_df["class_name"].value_counts() for name, loader in loaders.items()})

## Mô hình

In [ ]:
from src.models.mobilenet import build_mobilenet_v3_small, count_parameters

model = build_mobilenet_v3_small(
    num_classes=pipeline_cfg["num_classes"],
    pretrained=train_cfg["pretrained"],
    dropout=train_cfg["dropout"],
)
params = count_parameters(model)
print(f"Tham số: {params['total']:,} tổng, {params['trainable']:,} được huấn luyện")

## Huấn luyện

In [ ]:
from src.training.trainer import fit

fit_result = fit(model, loaders, train_cfg, device, CLASSES, CHECKPOINT_PATH)

history = pd.DataFrame(fit_result["history"])
history.to_csv(HISTORY_PATH, index=False)
print(
    f"Epoch tốt nhất: {fit_result['best_epoch']}"
    f" | macro f1 val: {fit_result['best_macro_f1']:.4f}"
    f" | thời gian: {fit_result['train_seconds'] / 60:.1f} phút"
)

## Đường huấn luyện

In [ ]:
from src.evaluation.plots import plot_confusion_matrix, plot_history

fig = plot_history(history, CURVES_PATH)

## Đánh giá trên val

In [ ]:
from src.training.trainer import evaluate, load_checkpoint

best = load_checkpoint(model, CHECKPOINT_PATH, device)
val_result = evaluate(model, loaders["val"], device, CLASSES)

print(f"Checkpoint: epoch {best['epoch']}")
print(f"macro f1: {val_result['macro_f1']:.4f}")
print(f"balanced accuracy: {val_result['balanced_accuracy']:.4f}")
print(f"accuracy: {val_result['accuracy']:.4f}")

pd.Series(val_result["per_class_recall"], name="recall").to_frame()

## Ma trận nhầm lẫn

In [ ]:
fig = plot_confusion_matrix(val_result["confusion_matrix"], CLASSES, CONFUSION_PATH)

## Lưu kết quả

In [ ]:
record = {
    "run_name": RUN_NAME,
    "model": train_cfg["model"],
    "seed": SEED,
    "git_commit": git_commit(REPO_DIR),
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "hyperparameters": {
        **train_cfg,
        "batch_size": pipeline_cfg["batch_size"],
        "image_size": pipeline_cfg["image_size"],
        "optimizer": fit_result["optimizer"],
        "scheduler": fit_result["scheduler"],
    },
    "parameters": params,
    "best_epoch": fit_result["best_epoch"],
    "epochs_run": fit_result["epochs_run"],
    "train_seconds": round(fit_result["train_seconds"], 1),
    "environment": describe_device(device),
    "checkpoint": str(CHECKPOINT_PATH.relative_to(REPO_DIR)),
    "val": {
        key: val_result[key]
        for key in ("macro_f1", "balanced_accuracy", "accuracy", "per_class_recall", "confusion_matrix")
    },
}
RESULT_PATH.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Đã lưu: {RESULT_PATH.relative_to(REPO_DIR)}")